# Aperture shapes

In [ ]:
import numpy as np
from photutils.aperture import CircularAperture, RectangularAnnulus
from photutils.datasets import make_100gaussians_image, make_model_image, make_model_params, make_noise_image
from photutils.psf import CircularGaussianPSF

from matplotlib import pyplot as plt

plt.style.use('../photutils_notebook_style.mplstyle')

## Shapes built in to [`photutils`](https://photutils.readthedocs.io/en/stable/) 

The following shapes are built in to `photutils`:

<a><img src="apertures.png" width=700 alt="Examples of circular, elliptical, and rectangular apertures and annuli."></a>

Each of these can be defined either in pixel coordinates or in celestial coordinates (using a WCS transformation).

It is also possible for users to create custom aperture shapes.

### Creating apertures

Apertures are created from one or more positions, which is the center of each aperture, and a size. The details of what the size entails depends on the apaerture shape. For a circular aperture, for example, the size is simply the radius of the aperture. A rectangular aperture requires a width, height and orientation, while an elliptical aperture requires a semimajor and semiminor axis an orientation. Annuli require both an inner size and an outer size.

Regardless of the way the size is specified, *each aperture has a single size*, though a single aperture can have multiple positions.

As an example, we create a small test image with a handful of stars on which we place some apertures. We construct the image first, with 10 reasonably bright (meaning well above the image background) stars.

In [ ]:
model = CircularGaussianPSF()

shape = (200, 200)
n_stars = 5
avg_fwhm = 4

# The positions of the model stars are in the columns x_0, y_0, and the FWHM
# is in the columns 
model_stars = make_model_params(
    shape, n_stars,
    min_separation=40,
    flux=(500, 2500),
    fwhm=(avg_fwhm - 0.1, avg_fwhm + 0.1), 
    seed=895437
)

model_shape = np.array(shape) // 10
image = (
    make_model_image(shape, model, model_stars, model_shape=model_shape, x_name='x_0', y_name='y_0')
    + make_noise_image(shape, mean=10, stddev=5, seed=4321)
)

Next, we make a circular aperture and rectangular annulus around each star. There is no compelling reason to use a rectangular annulus for actual photometry; it is done here simply to illustrate their creation.

Note that a single aperture or annulus object can have multliple positions but only a single size.

In [ ]:
xy_pairs = [(x, y) for x, y in model_stars["x_0", "y_0"]]
circles = CircularAperture(xy_pairs, 1.5 * avg_fwhm)

rectangles = RectangularAnnulus(
    xy_pairs, 
    w_in=2 * 3 * avg_fwhm, w_out=2 * 5 * avg_fwhm,
    h_in=2 * 1.75 * avg_fwhm, h_out=2 * 3 * avg_fwhm,
    theta=20
)

In [ ]:
fig, ax = plt.subplots()
circles.plot(ax=ax, color="red")
rectangles.plot(ax=ax, color="yellow")
ax.imshow(image, origin='lower')

### Creating apertures using sky coordinates

WORDS HERE SOON!

In [ ]:
# Code here soon!

## Using a region as an aperture

The main reason to use an Astropy region using the [`regions` package](https://astropy-regions.readthedocs.io/en/stable/index.html) is that `regions` can read, create, and represent regions created by ds9 and other software; see this [documentation for doing so](https://astropy-regions.readthedocs.io/en/stable/region_io.html).

Note that not all regions are supported -- the only regions that are supported are those that can be converted to one of the built-in aperture shapes. See the [documentation for `region_to_aperture`](https://photutils.readthedocs.io/en/stable/api/photutils.aperture.region_to_aperture.html#photutils.aperture.region_to_aperture) for details.

## Creating custom aperture shapes

If you need an aperture shape that is not built in to [`photutils`](https://photutils.readthedocs.io/en/stable/) then you need to create your own aperture subclass as [described here](https://photutils.readthedocs.io/en/stable/user_guide/aperture.html#defining-your-own-custom-apertures).